In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_train, load_test, remove_outliers, split_xy
from src.cleaning import clean

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


In [2]:
train_df = remove_outliers(load_train())
print(len(load_train()), "->", len(train_df), "rows after removing outliers") # 1460 -> 1458 on the real data

X, y = split_xy(train_df)
cleaned = clean(X)

before = X.isna().sum()
after = cleaned.isna().sum()
gaps = pd.DataFrame({"before": before, "after": after.reindex(before.index)})
gaps[gaps["before"] > 0].sort_values("before", ascending=False).head(15)

1460 -> 1458 rows after removing outliers


,before,after
PoolQC,1452,0.0
MiscFeature,1404,0.0
Alley,1367,0.0
Fence,1177,0.0
MasVnrType,872,0.0
FireplaceQu,690,0.0
LotFrontage,259,259.0
GarageType,81,0.0
GarageYrBlt,81,NaN
GarageFinish,81,0.0


In [3]:
from src.features import add_features

fe = add_features(cleaned)
new_cols = [c for c in fe.columns if c not in cleaned.columns]
print(new_cols)

check = fe[new_cols + ["OverallQual", "GrLivArea"]].assign(LogPrice=y.values)
check.corr()["LogPrice"].sort_values(ascending=False).round(2)


['TotalSF', 'TotalBath', 'TotalPorchSF', 'HouseAge', 'RemodAge', 'IsRemodeled', 'IsNew', 'HasPool', 'HasGarage', 'HasBasement', 'HasFireplace', 'Has2ndFloor', 'QualSF']


LogPrice        1.00
QualSF          0.88
TotalSF         0.83
OverallQual     0.82
GrLivArea       0.73
TotalBath       0.68
HasFireplace    0.51
TotalPorchSF    0.40
HasGarage       0.32
IsNew           0.22
HasBasement     0.20
Has2ndFloor     0.15
HasPool         0.08
IsRemodeled    -0.07
RemodAge       -0.57
HouseAge       -0.59
Name: LogPrice, dtype: float64

In [4]:
from sklearn.linear_model import Ridge
from src.pipeline import build_pipeline

pipe = build_pipeline(Ridge(alpha=10)).fit(X, y)
Z = pipe[:-1].transform(X) # everything except the final model
print(Z.shape, "NaN left:", int(np.isnan(Z).sum()))

names = pipe.named_steps["prep"].get_feature_names_out()
print(len(names), "columns, e.g.", list(names[:4]), "...", list(names[-2:]))

skewed = pipe.named_steps["prep"].named_transformers_["num"].named_steps["skew"].skewed_
print(len(skewed), "numeric columns got a log transform")

(1458, 224) NaN left: 0
224 columns, e.g. ['num__LotFrontage', 'num__LotArea', 'num__LotShape', 'num__LandSlope'] ... ['cat__SaleCondition_Partial', 'cat__SaleCondition_infrequent_sklearn']
36 numeric columns got a log transform


In [5]:
from sklearn.dummy import DummyRegressor
from src.cv import cv_rmse, CV, rmse

results = {}
def evaluate(name, model):
 mean, std = cv_rmse(model, X, y)
 results[name] = (mean, std)
 print(f"{name:<24} RMSE {mean:.4f} +/- {std:.4f}")
 
evaluate("baseline (always the mean)", build_pipeline(DummyRegressor(strategy="mean")))
evaluate("ridge alpha=10", build_pipeline(Ridge(alpha=10)))

baseline (always the mean) RMSE 0.3993 +/- 0.0168
ridge alpha=10           RMSE 0.1136 +/- 0.0077


In [6]:
test = load_test()
X_test = test.drop(columns="Id")

ridge = build_pipeline(Ridge(alpha=10)).fit(X, y) # fit on ALL training rows
quick = pd.DataFrame({"Id": test["Id"], "SalePrice": np.expm1(ridge.predict(X_test))}) # back to dollars
quick.to_csv("../submissions/submission_ridge.csv", index=False)
quick.head()

,Id,SalePrice
0,1461,118020.144261
1,1462,152590.431147
2,1463,178036.167993
3,1464,199720.685737
4,1465,200165.407042


In [7]:
from sklearn.linear_model import LinearRegression
evaluate("linear regression (no penalty)", build_pipeline(LinearRegression()))
for a in [1, 3, 10, 30, 100]:
 evaluate(f"ridge alpha={a}", build_pipeline(Ridge(alpha=a)))

linear regression (no penalty) RMSE 0.1872 +/- 0.1256
ridge alpha=1            RMSE 0.1157 +/- 0.0078
ridge alpha=3            RMSE 0.1141 +/- 0.0077
ridge alpha=10           RMSE 0.1136 +/- 0.0077
ridge alpha=30           RMSE 0.1143 +/- 0.0079
ridge alpha=100          RMSE 0.1171 +/- 0.0084


In [8]:
plain = build_pipeline(LinearRegression()).fit(X, y)
ridge10 = build_pipeline(Ridge(alpha=10)).fit(X, y)
print("largest |weight|, no penalty:", round(float(np.abs(plain.named_steps["model"].coef_).max()), 3))
print("largest |weight|, ridge :", round(float(np.abs(ridge10.named_steps["model"].coef_).max()), 3))

largest |weight|, no penalty: 5.716
largest |weight|, ridge : 0.137


In [9]:
from src.pipeline import make_models
models = make_models()
for name in ["ridge", "lasso", "elasticnet"]:
 evaluate(name, models[name])


ridge                    RMSE 0.1132 +/- 0.0077
lasso                    RMSE 0.1115 +/- 0.0067
elasticnet               RMSE 0.1115 +/- 0.0067


In [ ]:
lasso = models["lasso"].fit(X, y)
coefs = pd.Series(lasso.named_steps["model"].coef_,
 index=lasso.named_steps["prep"].get_feature_names_out())
print("chosen alpha:", round(float(lasso.named_steps["model"].alpha_), 5))
print((coefs != 0).sum(), "of", len(coefs), "features kept")

pd.concat([coefs.sort_values().head(8), coefs.sort_values().tail(8)]).round(3)

chosen alpha: 0.00033
118 of 224 features kept


cat__MSZoning_C (all)                -0.313
num__HouseAge                        -0.074
num__KitchenAbvGr                    -0.068
num__HasBasement                     -0.056
cat__SaleCondition_Abnorml           -0.043
cat__Condition1_RRAe                 -0.043
cat__GarageType_infrequent_sklearn   -0.042
cat__Neighborhood_Edwards            -0.030
cat__Exterior1st_BrkFace              0.061
cat__Neighborhood_NridgHt             0.061
cat__Neighborhood_NoRidge             0.064
cat__Neighborhood_StoneBr             0.102
num__OverallQual                      0.104
cat__Neighborhood_Crawfor             0.107
num__GrLivArea                        0.135
num__Functional                       0.198
dtype: float64